# 01 — RCT 数据探索

对 RCT（Robotic Contact Tactile）数据集做统计质量评估的初步探索：
序列规模、帧数分布、材料类别分布、触觉图像统计、接触力时序、按类别分层统计，
以及关于数据均衡性与异常序列的关键发现。

复用入口：`tactile_qc.io.load_rct_sequences`。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from tactile_qc.io import load_rct_sequences, FORCE_COLUMNS

pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)

DATA_DIR = Path('D:\\Desktop\\Data\\项目类\\2026.09.13——Tactile Data Quality Inspector\\data\\rct\\rct_dataset\\rct_dataset')
print("data dir:", DATA_DIR, "exists:", DATA_DIR.exists())


data dir: D:\Desktop\Data\项目类\2026.09.13——Tactile Data Quality Inspector\data\rct\rct_dataset\rct_dataset exists: True


In [2]:
# 加载全部序列
ds = load_rct_sequences(DATA_DIR)
n_materials = len({s.material_id for s in ds})
n_seq = len(ds)
n_frames = sum(len(s) for s in ds)
n_force = sum(s.has_force for s in ds)
print(f"materials : {n_materials}  (README: 122)")
print(f"sequences : {n_seq}  (README: 1832)")
print(f"frames    : {n_frames}  (README: 29279)")
print(f"with force: {n_force}  (missing: {n_seq - n_force}, README: 5)")

materials : 122  (README: 122)
sequences : 1832  (README: 1832)
frames    : 29279  (README: 29279)
with force: 1827  (missing: 5, README: 5)


## 1. 序列数量与帧数分布

In [3]:
summary = ds.summary_dataframe()
print("per-sequence summary (head):")
summary.head()

per-sequence summary (head):


,sequence_id,material_id,material_label,position,sensor,n_frames,depth_min,depth_max,has_force,force_trace_len,force_mag_max
0,material_0700000_position_1_sensor_1,0700000,Plastic_Rubber,1,1,17,105.5,107.0,True,27,1.278202
1,material_0700000_position_1_sensor_2,0700000,Plastic_Rubber,1,2,16,105.5,107.0,True,16,0.000000
2,material_0700000_position_1_sensor_3,0700000,Plastic_Rubber,1,3,16,105.5,107.0,True,16,0.123693
3,material_0700000_position_2_sensor_1,0700000,Plastic_Rubber,2,1,16,105.5,107.0,True,16,0.389230
4,material_0700000_position_2_sensor_2,0700000,Plastic_Rubber,2,2,16,105.5,107.0,True,16,0.308707


In [4]:
desc = summary["n_frames"].describe()
print(desc)
fig = go.Figure(go.Histogram(x=summary["n_frames"], nbinsx=40))
fig.update_layout(title="每个序列的触觉帧数分布",
                  xaxis_title="contact frames / sequence", bargap=0.02)
fig.show()

count    1832.000000
mean       15.981987
std         0.979976
min         1.000000
25%        16.000000
50%        16.000000
75%        16.000000
max        17.000000
Name: n_frames, dtype: float64


## 2. 材料类别分布

In [5]:
cat_seq = summary.groupby("material_label").agg(
    n_materials=("material_id", "nunique"),
    n_sequences=("sequence_id", "count"),
    n_frames=("n_frames", "sum"),
).sort_values("n_sequences", ascending=False)
cat_seq

,n_materials,n_sequences,n_frames
material_label,,,
Paper_Cardboard,45,613,9830
Plastic_Rubber,32,471,7544
Metal,20,376,5936
Textiles_Leather,10,147,2357
Wood_Bamboo_Cork,7,105,1688
Crafts,4,63,1011
Small_Items,3,42,673
Unknown,1,15,240


In [6]:
fig = go.Figure()
fig.add_trace(go.Bar(x=cat_seq.index, y=cat_seq["n_materials"], name="materials"))
fig.add_trace(go.Bar(x=cat_seq.index, y=cat_seq["n_sequences"], name="sequences"))
fig.update_layout(title="各材料类别的材料数 / 序列数",
                  barmode="group", xaxis_tickangle=-30)
fig.show()

## 3. 触觉图像基本统计

逐序列流式统计全局像素分布（均值 / 方差 / 亮度直方图 / 饱和帧）。
全量遍历 29,279 帧触觉深度图。

In [7]:
total_pix = 0
sum_v = 0
sumsq_v = 0
global_min = 255
global_max = 0
hist = np.zeros(256, dtype=np.int64)
per_frame_mean = []
n_sat = 0  # 完全饱和(全 255)的帧数
for i, s in enumerate(ds, 1):
    arr = s.load_frames()                       # (N,H,W,3) uint8
    flat = arr.reshape(-1)
    total_pix += flat.size
    sum_v += int(flat.sum())
    sumsq_v += int(np.square(flat.astype(np.int64)).sum())
    global_min = min(global_min, int(flat.min()))
    global_max = max(global_max, int(flat.max()))
    hist += np.bincount(flat, minlength=256)[:256]
    per_frame_mean.append(arr.mean(axis=(1, 2, 3)))
    n_sat += int((arr >= 255).all(axis=(1, 2, 3)).sum())
    if i % 200 == 0:
        print(f"  processed {i}/{len(ds)} sequences", flush=True)

mean_v = sum_v / total_pix
var_v = sumsq_v / total_pix - mean_v ** 2
pfm = np.concatenate(per_frame_mean)
print(f"global pixel mean = {mean_v:.2f}, std = {var_v**0.5:.2f}")
print(f"global min = {global_min}, max = {global_max}")
print(f"per-frame mean: min={pfm.min():.2f} max={pfm.max():.2f}")
print(f"fully saturated frames (all=255): {n_sat}")

  processed 200/1832 sequences


  processed 400/1832 sequences


  processed 600/1832 sequences


  processed 800/1832 sequences


  processed 1000/1832 sequences


  processed 1200/1832 sequences


  processed 1400/1832 sequences


  processed 1600/1832 sequences


  processed 1800/1832 sequences


global pixel mean = 111.47, std = 37.38
global min = 0, max = 255
per-frame mean: min=98.83 max=154.45
fully saturated frames (all=255): 0


In [8]:
fig = go.Figure(go.Bar(x=np.arange(256), y=hist))
fig.update_layout(title="触觉图像像素亮度分布（全量, log）",
                  xaxis_title="pixel value", yaxis_title="count", yaxis_type="log")
fig.show()

## 4. 接触力时序可视化

每类材料取一个序列，绘制力幅值随压入深度 z 的变化曲线。

In [9]:
# 每个材料类别取第一个有力数据的序列
sample = []
seen = set()
for s in ds:
    if s.material_label not in seen and s.has_force:
        sample.append(s)
        seen.add(s.material_label)

fig = go.Figure()
for s in sample:
    mag = s.force_magnitude()
    fig.add_trace(go.Scatter(x=s.z_positions, y=mag, mode="lines",
                            name=s.material_label))
fig.update_layout(title="力幅值 ||F_ext|| 随压入深度 z 的变化（每类一个序列）",
                  xaxis_title="z position", yaxis_title="||F_ext|| (N)")
fig.show()

## 5. 按材料类别的分层统计

In [10]:
cat_force = summary.dropna(subset=["force_mag_max"]).groupby("material_label").agg(
    n_sequences=("force_mag_max", "count"),
    force_max_mean=("force_mag_max", "mean"),
    force_max_std=("force_mag_max", "std"),
).sort_values("force_max_mean", ascending=False)
cat_force

,n_sequences,force_max_mean,force_max_std
material_label,,,
Paper_Cardboard,613,1.125902,3.324174
Textiles_Leather,147,1.094463,3.946370
Plastic_Rubber,470,1.059965,3.762308
Wood_Bamboo_Cork,105,0.944606,3.066847
Unknown,15,0.706996,0.569790
Small_Items,42,0.523839,0.822428
Metal,372,0.470537,0.702009
Crafts,63,0.331156,0.238065


In [11]:
fig = go.Figure(go.Bar(
    x=cat_force.index,
    y=cat_force["force_max_mean"],
    error_y=dict(type="data", array=cat_force["force_max_std"]),
))
fig.update_layout(title="各类别峰值力的均值（误差条=序列间标准差）",
                  xaxis_tickangle=-30, yaxis_title="peak |F_ext| (N)")
fig.show()

## 6. 关键发现

- **数据是否均衡？**
- **是否存在明显的异常序列？**

In [12]:
no_force = summary[~summary["has_force"]]
q_hi = summary["n_frames"].quantile(0.99)
frame_outliers = summary[(summary["n_frames"] < 5) | (summary["n_frames"] > q_hi)]
flat_force = summary[summary["has_force"] & (summary["force_mag_max"] < 0.5)]

print(f"无力轨迹的序列: {len(no_force)}")
if len(no_force):
    print(no_force[["sequence_id", "material_id", "position", "sensor"]].to_string(index=False))
print(f"\n帧数异常序列 (<5 或 >99%分位={q_hi:.0f}): {len(frame_outliers)}")
if len(frame_outliers):
    print(frame_outliers[["sequence_id", "n_frames"]].to_string(index=False))
print(f"\n峰值力 <0.5N 的平坦力曲线序列: {len(flat_force)}")
if len(flat_force):
    print(flat_force[["sequence_id", "force_mag_max"]].to_string(index=False))

无力轨迹的序列: 5
                         sequence_id material_id  position  sensor
material_0700070_position_1_sensor_3     0700070         1       3
material_0701490_position_4_sensor_3     0701490         4       3
material_0701490_position_8_sensor_3     0701490         8       3
material_701440_position_10_sensor_2      701440        10       2
material_701440_position_10_sensor_3      701440        10       3

帧数异常序列 (<5 或 >99%分位=17): 7
                         sequence_id  n_frames
material_0700070_position_1_sensor_3         1
material_0701490_position_4_sensor_3         1
material_0701490_position_8_sensor_3         1
 material_701440_position_1_sensor_1         1
material_701440_position_10_sensor_2         1
material_701440_position_10_sensor_3         1
 material_701450_position_6_sensor_1         1

峰值力 <0.5N 的平坦力曲线序列: 1306
                          sequence_id  force_mag_max
 material_0700000_position_1_sensor_2       0.000000
 material_0700000_position_1_sensor_3       0.12369

### 结论

**数据规模**：与 README 完全吻合——122 材料 / 1832 序列 / 29279 帧触觉图 / 1827 条力轨迹（缺失 5 条），说明 `load_rct_sequences` 的解析逻辑可信。

**类别失衡（显著）**：7 个大类样本量差异大。Paper_Cardboard + Plastic_Rubber 合计占 77 材料 / 1084 序列 / 17374 帧；Small_Items 仅 3 材料 / 42 序列。按序列计的失衡比按材料计更严重，后续建模/评估需做类别加权或分层抽样。另：材料 id `070020` 在 `material_categories.json` 中无类别映射，被标记为 Unknown，需补全。

**帧数**：每序列 16–17 帧（均值 15.98，std 0.98），非常稳定；仅 7 条退化为单帧（material_0700070 / 0701490 / 701440 / 701450 的个别 position×sensor），其中 5 条正是缺失力数据的轨迹。

**图像统计**：640×480 RGB，全局均值 111.5、std 37.4，min=0 max=255，**无完全饱和帧**；亮度直方图呈典型双峰（背景区 + 接触区），图像层健康。

**接触力（关键发现）**：峰值外力 ||F_ext|| 强烈依赖材料刚度——刚性材料 Metal 82% / Crafts 90% 的序列峰值力 <0.5N，软质材料 65–74%；整体 1306/1827 (71%) 序列峰值外力 <0.5N。即「外力幅值」作为全局质量信号偏弱、且被刚度主导。建议后续 quality / anomaly 阶段改用 raw force、接触相 Fz 或力-深度曲线斜率等更稳健的特征，而非峰值外力幅值。

**异常序列**：5 条缺力轨迹（与 README 明列一致：0700070 p1/s3、0701490 p4/s3 & p8/s3、701440 p10/s2 & p10/s3）；7 条单帧退化序列；1 个类别未映射材料（070020）。其余序列结构与图像均正常。